In [6]:
import pandas as pd
import os

BASE_DIR = "/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/Project 1/Comps"
BASE_FILE = "AktAcc_gemini_abl2_40001018_43012016_20260225.csv"
COMPARE_FILE = "AktAch_gemini_abl1_40001018_43012016_20260306.csv"
OUTPUT_DIR = "/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/Project 1/Comps/Results"

COLS = [
    "annotation",
]
KEY_COL = "TokenID"

os.makedirs(OUTPUT_DIR, exist_ok=True)

base = pd.read_csv(os.path.join(BASE_DIR, BASE_FILE)).set_index(KEY_COL)[COLS].sort_index()
comp = pd.read_csv(os.path.join(BASE_DIR, COMPARE_FILE)).set_index(KEY_COL)[COLS].sort_index()

shared = base.index.intersection(comp.index)
if len(shared) < len(base.index):
    print(f"Note: {len(base.index) - len(shared)} base row(s) and {len(comp.index) - len(shared)} comp row(s) skipped (no matching TokenID)")
base_aligned = base.loc[shared]
comp_aligned = comp.loc[shared]

match = (base_aligned.astype(int) == comp_aligned.astype(float)).replace({True: "Y", False: "N"})
base_renamed = base_aligned.rename(columns={c: f"{c}_KD" for c in COLS})
comp_renamed = comp_aligned.rename(columns={c: f"{c}_BB" for c in COLS})

interleaved_cols = [col for c in COLS for col in (f"{c}_KD", f"{c}_BB", c)]
result = pd.concat([base_renamed, comp_renamed, match], axis=1)[interleaved_cols]

result.reset_index().to_csv(os.path.join(OUTPUT_DIR, "gemini_abl1_aktach.csv"), index=False)
print("Done: feature_comp")

Done: feature_comp
